# 🎈 Agesis EYE — Stage 03: YOLOv8 Black Balloon Model Training

This notebook trains a high-precision, low-latency **YOLOv8 Nano** model to detect black balloons from an ESP32-CAM stream.

### 🎯 Objectives & Rules
- **Target Class**: `0: balloon`
- **Architecture**: `yolov8n.pt` (fine-tuned from pretrained COCO weights)
- **Resolution**: `imgsz=320` (matches 320x240 QVGA camera stream 1:1)
- **Precision Target**: $\ge 0.97$ (crucial for safe laser targeting — zero false fires)
- **Recall Target**: $\ge 0.90$
- **Export**: `best.onnx` for lightweight, real-time laptop inference (Stage 04)

In [ ]:
# 1. Verify GPU Acceleration & Install Ultralytics
!nvidia-smi
!pip install ultralytics -q

import os
import sys
import glob
import yaml
import torch
from IPython.display import Image, display
from ultralytics import YOLO

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")

## 2. Dataset Setup & Path Resolution

Automatically finds `data.yaml` whether running on:
1. **Kaggle** (`/kaggle/input/...`)
2. **Google Colab** (`/content/...`)
3. **Local Machine**

In [ ]:
# Auto-detect data.yaml location
possible_paths = [
    "/kaggle/input/*/data.yaml",
    "/kaggle/input/*/*/data.yaml",
    "/content/dataset/data.yaml",
    "dataset/data.yaml",
    "../dataset/data.yaml"
]

yaml_path = None
for pattern in possible_paths:
    matches = glob.glob(pattern)
    if matches:
        yaml_path = os.path.abspath(matches[0])
        break

if yaml_path is None:
    # If dataset is uploaded as a zip in the working directory
    zip_files = glob.glob("*.zip") + glob.glob("/kaggle/input/*/*.zip")
    if zip_files:
        print(f"Extracting {zip_files[0]}...")
        !unzip -q "{zip_files[0]}" -d dataset
        yaml_path = os.path.abspath("dataset/data.yaml")

print(f"[+] Using data.yaml at: {yaml_path}")

# Ensure path inside data.yaml points to dataset root directory
dataset_root = os.path.dirname(yaml_path)
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = dataset_root
working_yaml = os.path.abspath("data_train.yaml")
with open(working_yaml, 'w') as f:
    yaml.dump(data_cfg, f)

print(f"[+] Configured training YAML: {working_yaml}")
with open(working_yaml) as f:
    print(f.read())

## 3. Train YOLOv8 Nano Model

- Fine-tuning from pretrained COCO weights (`yolov8n.pt`)
- `imgsz=320` to match camera stream directly
- `patience=25` early stopping
- `fliplr=0.5` horizontal reflection, `flipud=0.0` (balloons don't float upside-down)
- `mosaic=1.0` for multi-scale robustness

In [ ]:
# Load pretrained model
model = YOLO("yolov8n.pt")

# Train
results = model.train(
    data=working_yaml,
    epochs=80,
    imgsz=320,
    batch=16,
    patience=25,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    project='runs',
    name='agesis_balloon_v1',
    plots=True,
    verbose=True
)

print("\n[+] Training finished successfully!")

## 4. Evaluate on Held-out Test Split

Stage 03 Rule: **Judge solely by the held-out test split, never training metrics.**

In [ ]:
# Validate on held-out TEST set
best_weights = "runs/agesis_balloon_v1/weights/best.pt"
eval_model = YOLO(best_weights)

metrics = eval_model.val(data=working_yaml, split='test')

precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

print(f"\n{'=' * 45}")
print(f"  HELD-OUT TEST SET EVALUATION")
print(f"{'=' * 45}")
print(f"  Precision (target >= 0.97) : {precision:.4f} {'[PASS]' if precision >= 0.97 else '[REVIEW]'}")
print(f"  Recall    (target >= 0.90) : {recall:.4f} {'[PASS]' if recall >= 0.90 else '[REVIEW]'}")
print(f"  mAP@50                     : {map50:.4f}")
print(f"  mAP@50-95                  : {map50_95:.4f}")
print(f"{'=' * 45}")

## 5. Visual Inspection of Training Curves & Confusion Matrix

In [ ]:
# Display Confusion Matrix & Loss Plots
plots_to_show = [
    "runs/agesis_balloon_v1/results.png",
    "runs/agesis_balloon_v1/confusion_matrix.png",
    "runs/agesis_balloon_v1/val_batch0_pred.jpg"
]

for p in plots_to_show:
    if os.path.exists(p):
        print(f"\n--- {os.path.basename(p)} ---")
        display(Image(filename=p))

## 6. Export to ONNX for Laptop Inference (Stage 04)

ONNX allows ultra-fast CPU inference on your laptop without requiring PyTorch/CUDA runtime.

In [ ]:
# Export best model to ONNX
onnx_path = eval_model.export(format="onnx", imgsz=320, dynamic=False, simplify=True)
print(f"[+] Exported ONNX model: {onnx_path}")

# Package weights for download
!zip -j agesis_model_weights.zip runs/agesis_balloon_v1/weights/best.pt runs/agesis_balloon_v1/weights/best.onnx

print(f"\n{'=' * 50}")
print(f"  Model training & export complete!")
print(f"  Download 'agesis_model_weights.zip' to use in Stage 04")
print(f"{'=' * 50}")